# 社交高尔夫问题

**类别：** 调度

来源： [https://www.hexaly.com/templates/social-golfer-problem](https://www.hexaly.com/templates/social-golfer-problem)


## 问题

**在 社交高尔夫问题 中**，我们考虑一个拥有 32 名会员的高尔夫俱乐部。每位会员每周打一次高尔夫，总是 4 人一组。该问题的目标是构建一个为期 10 周的高尔夫会员日程表，使社交最大化，即尽可能减少重复的同组。更一般地，问题是对 m 组 n 名高尔夫爱好者在 p 周内进行调度，以最大化社交。关于更多细节，请参阅 [CSPLib](http://www.csplib.org/Problems/prob010/) 或 [MathPuzzle](http://www.mathpuzzle.com/MAA/54-Golf%20Tournaments/mathgames_08_14_07.html)。

	

### 学到的建模原则

- 添加 [布尔决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#boolean-decisions) 来建模日程
- 使用 [`非线性算子`](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来计算每对高尔夫爱好者之间的同组次数
- 了解 Hexaly Optimizer 的建模风格：[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)


## 数据

我们提供的 社交高尔夫问题 实例文件包含三个数字：

- 组数
- 每组的大小
- 周数

我们在示例中使用的实例已知存在无重复同组的解。


## 模型

社交高尔夫问题 的 Hexaly 模型使用 布尔决策变量。对于每周 w、每位高尔夫爱好者 gf 以及每组 gr，x[w][gr][gf] 等于 1 表示高尔夫爱好者 gf 在第 w 周位于 gr 组，否则为 0。使用 ‘sum’ 算子，我们约束每位高尔夫爱好者每周恰好被分到一个组。我们还确保每个组的大小正确。

使用 **and** 算子，我们计算中间表达式，以确定每周以及每对高尔夫爱好者是否在该周相遇。然后我们可以使用这些表达式来计算每对高尔夫爱好者之间的相遇总次数。最后，借助 **max** 算子，我们可以计算要最小化的目标函数，即赛季中任何一对高尔夫爱好者相遇次数的最大值。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

if len(sys.argv) < 2:
    print("Usage: python social_golfer.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)


def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]


with hexaly.optimizer.HexalyOptimizer() as optimizer:
    #
    # Read instance data
    #
    file_it = iter(read_integers(sys.argv[1]))
    nb_groups = next(file_it)
    group_size = next(file_it)
    nb_weeks = next(file_it)
    nb_golfers = nb_groups * group_size

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # Decision variables
    # 0-1 decisions variables: x[w][gr][gf]=1 if golfer gf is in group gr on week w
    x = [[[model.bool() for gf in range(nb_golfers)]
          for gr in range(nb_groups)] for w in range(nb_weeks)]

    # Each week, each golfer is assigned to exactly one group
    for w in range(nb_weeks):
        for gf in range(nb_golfers):
            model.constraint(
                model.eq(model.sum(x[w][gr][gf] for gr in range(nb_groups)), 1))

    # Each week, each group contains exactly group_size golfers
    for w in range(nb_weeks):
        for gr in range(nb_groups):
            model.constraint(
                model.eq(model.sum(x[w][gr][gf] for gf in range(nb_golfers)), group_size))

    # Golfers gf0 and gf1 meet in group gr on week w if both are
    # assigned to this group for week w
    meetings = [None] * nb_weeks
    for w in range(nb_weeks):
        meetings[w] = [None] * nb_groups
        for gr in range(nb_groups):
            meetings[w][gr] = [None] * nb_golfers
            for gf0 in range(nb_golfers):
                meetings[w][gr][gf0] = [None] * nb_golfers
                for gf1 in range(gf0 + 1, nb_golfers):
                    meetings[w][gr][gf0][gf1] = model.and_(x[w][gr][gf0], x[w][gr][gf1])

    # The number of meetings of golfers gf0 and gf1 is the sum
    # of their meeting variables over all weeks and groups
    redundant_meetings = [None] * nb_golfers
    for gf0 in range(nb_golfers):
        redundant_meetings[gf0] = [None] * nb_golfers
        for gf1 in range(gf0 + 1, nb_golfers):
            nb_meetings = model.sum(meetings[w][gr][gf0][gf1] for w in range(nb_weeks)
                                    for gr in range(nb_groups))
            redundant_meetings[gf0][gf1] = model.max(nb_meetings - 1, 0)

    # the goal is to minimize the number of redundant meetings
    obj = model.sum(redundant_meetings[gf0][gf1] for gf0 in range(nb_golfers)
                    for gf1 in range(gf0 + 1, nb_golfers))
    model.minimize(obj)

    model.close()

    # Parameterize the optimizer
    optimizer.param.nb_threads = 1
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 10

    optimizer.solve()

    #
    # Write the solution in a file with the following format:
    # - the objective value
    # - for each week and each group, write the golfers of the group
    # (nb_weeks x nbGroupes lines of group_size numbers).
    #
    if len(sys.argv) >= 3:
        with open(sys.argv[2], 'w') as f:
            f.write("%d\n" % obj.value)
            for w in range(nb_weeks):
                for gr in range(nb_groups):
                    for gf in range(nb_golfers):
                        if x[w][gr][gf].value:
                            f.write("%d " % (gf))
                    f.write("\n")
                f.write("\n")
